# Resume Dataset EDA + Baseline Model (200k кандидатов)

**Цель:** исследовать факторы влияющие на найм, построить рейтинг предикторов через IC-анализ и проверить предсказательную силу через ML модель.

**Структура:**
1. Загрузка и первичный осмотр
2. Очистка данных
3. Распределения и выбросы
4. Анализ зависимостей
5. IC анализ — рейтинг факторов найма
6. Корреляционная матрица
7. Анализ University Tier
8. Baseline модель (Random Forest)
9. Диагностика overfit и регуляризация
10. Итоговые выводы

## 1. Загрузка и первичный осмотр

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('resume_dataset_200k_enhanced.csv')

print('Размер:', df.shape)
print('\nТипы данных и пропуски:')
df.info()
print('\nДубликатов:', df.duplicated().sum())

In [ ]:
df.head()

In [ ]:
print('Базовая статистика (до очистки):')
df.describe().round(2)

**Наблюдения после первичного осмотра:**
- 200k строк, пропусков нет, дубликатов нет
- `university_tier` — строка ("Tier 1"), требует конвертации в число
- `hired` — бинарная целевая переменная (0/1)
- `cgpa` имеет значения > 10 — физически невозможно, требует clip
- `resume_length_words` может иметь отрицательные значения — требует clip

## 2. Очистка данных

In [ ]:
df_clean = df.copy()

# 1. university_tier: "Tier 1" → 1
df_clean['university_tier'] = (df_clean['university_tier']
    .str.replace('Tier', '', regex=False)
    .str.replace(' ', '', regex=False)
    .astype(int))

# 2. cgpa: физически невозможны значения > 10
df_clean['cgpa'] = df_clean['cgpa'].clip(upper=10)

# 3. resume_length_words: отрицательная длина невозможна
df_clean['resume_length_words'] = df_clean['resume_length_words'].clip(lower=0)

print('Типы данных после очистки:')
print(df_clean.dtypes)
print(f'\nBалансировка классов hired:')
print(df_clean['hired'].value_counts(normalize=True).round(4))

**Что исправлено:**
- `university_tier`: "Tier 1" → 1, "Tier 2" → 2, "Tier 3" → 3
- `cgpa`: clip(upper=10) — физически невозможные значения обрезаны
- `resume_length_words`: clip(lower=0) — отрицательная длина невозможна
- **Дисбаланс классов:** hired=1 (~70%), hired=0 (~30%) — умеренный, учтём при построении модели

## 3. Распределения и выбросы

In [ ]:
numeric_cols = [
    'age', 'cgpa', 'internships', 'projects', 'programming_languages',
    'certifications', 'experience_years', 'hackathons', 'research_papers',
    'skills_score', 'soft_skills_score', 'resume_length_words', 'university_tier'
]  # candidate_id исключён — это ID, не фактор

fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df_clean[col], patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6))
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xticks([])

# Скрываем лишние ячейки
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Boxplot — поиск выбросов', fontsize=13)
plt.tight_layout()
plt.show()

**Наблюдения:**
- Выбросы в `internships`, `projects`, `hackathons`, `research_papers` — реальные активные кандидаты, **не удаляем**
- `skills_score` и `soft_skills_score` — правосторонний скос
- `hired` — бинарный (0/1), boxplot не информативен
- `cgpa` после clip(upper=10) — распределение нормализовалось

## 4. Анализ зависимостей

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# GPA vs Skills Score
yearly = df_clean.groupby('cgpa')['skills_score'].mean()
axes[0].plot(yearly.index, yearly.values, marker='o', linewidth=2, color='steelblue')
axes[0].set_title('GPA vs средний Skills Score')
axes[0].set_ylabel('Skills Score')
axes[0].set_xlabel('CGPA')
axes[0].grid(True)

# GPA vs P(hired)
bev_yearly = df_clean.groupby('cgpa')['hired'].mean()
axes[1].plot(bev_yearly.index, bev_yearly.values, marker='o', linewidth=2, color='green')
axes[1].set_title('GPA vs вероятность найма')
axes[1].set_ylabel('P(hired)')
axes[1].set_xlabel('CGPA')
axes[1].grid(True)

plt.tight_layout()
plt.show()

corr, p = spearmanr(df_clean['cgpa'], df_clean['skills_score'])
print(f'Spearman (cgpa → skills_score): {corr:.3f}, p-value: {p:.4f}')
print('Связь есть ✅' if p < 0.05 else 'Связи нет ❌')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Projects vs P(hired) — нелинейная зависимость
yearly = df_clean.groupby('projects')['hired'].mean()
axes[0].plot(yearly.index, yearly.values, marker='o', linewidth=2, color='steelblue')
axes[0].set_title('Projects vs вероятность найма')
axes[0].set_ylabel('P(hired)')
axes[0].set_xlabel('Projects')
axes[0].grid(True)

# Internships vs P(hired) — линейная зависимость
bev_yearly = df_clean.groupby('internships')['hired'].mean()
axes[1].plot(bev_yearly.index, bev_yearly.values, marker='o', linewidth=2, color='green')
axes[1].set_title('Internships vs вероятность найма')
axes[1].set_ylabel('P(hired)')
axes[1].set_xlabel('Internships')
axes[1].grid(True)

plt.tight_layout()
plt.show()

**Ключевые находки:**
- **GPA vs Skills:** r = 0.002, p = 0.47 — **полная независимость**. GPA не предсказывает навыки
- **GPA vs Hired:** слабая связь — работодатели смотрят на опыт, не оценки
- **Projects → Hired:** **нелинейный** эффект — при 10+ проектах P(hired) резко скачет до ~1.0
- **Internships → Hired:** **линейный** рост — каждая стажировка даёт +~3% к P(hired)
- ⚠️ Нелинейность означает что линейные модели не подойдут — нужен Random Forest или GBM

## 5. IC анализ — рейтинг факторов найма

**IC (Information Coefficient)** = Spearman корреляция между фактором и целевой переменной.
Показывает насколько хорошо фактор предсказывает найм.
Шкала: IC > 0.10 = сильный, 0.05–0.10 = рабочий, < 0.02 = нет сигнала.

In [ ]:
factors = [
    'cgpa', 'skills_score', 'soft_skills_score', 'internships',
    'projects', 'hackathons', 'certifications', 'research_papers',
    'programming_languages', 'experience_years', 'university_tier',
    'age', 'resume_length_words'
]

print(f'{"Фактор":<25} {"IC":>7}  {"p-value":>8}  {"Значимость"}')
print('-' * 58)
results = []
for col in factors:
    corr, p = spearmanr(df_clean[col], df_clean['hired'])
    sig = '✅' if p < 0.05 else '❌'
    results.append((col, corr, p, sig))

results.sort(key=lambda x: abs(x[1]), reverse=True)
for col, corr, p, sig in results:
    print(f'{col:<25} {corr:+.3f}   {p:.4f}   {sig}')

**Рейтинг факторов:**
- 🥇 `experience_years` IC=0.067 — главный предиктор найма
- 🥈 `internships` IC=0.047 — сильный сигнал
- 🥉 `skills_score` IC=0.045 — важен
- `projects`, `programming_languages` — умеренный сигнал
- `university_tier`, `age`, `research_papers` — **нулевой сигнал** (p > 0.05) → исключить из модели
- `cgpa` IC=0.012 — статистически значим, но практически бесполезен
- ⚠️ Все IC < 0.10 — датасет синтетический, сигнал слабый

## 6. Корреляционная матрица (Spearman)

In [ ]:
corr_matrix = df_clean[factors].corr(method='spearman')

plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Корреляционная матрица (Spearman)', fontsize=13)
plt.tight_layout()
plt.show()

**Ключевые корреляции:**
- `skills_score` ↔ `projects` = **0.70** — проекты развивают навыки
- `skills_score` ↔ `programming_languages` = **0.60** — больше языков = выше skills
- `skills_score` ↔ `certifications` = **0.29** — умеренная связь
- Остальные пары ~0.00 — факторы независимы ✅

**⚠️ Мультиколлинеарность:** `projects`, `programming_languages` и `skills_score` сильно коррелируют.
В линейной модели нельзя включать все три — дублируют информацию.
Random Forest устойчив к мультиколлинеарности.

## 7. Анализ University Tier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

tier_hired = df_clean.groupby('university_tier')['hired'].mean()
axes[0].bar(tier_hired.index, tier_hired.values, color='steelblue', edgecolor='black')
axes[0].set_title('Вероятность найма по University Tier')
axes[0].set_xlabel('University Tier')
axes[0].set_ylabel('P(hired)')
axes[0].set_xticks(tier_hired.index)

tier_gpa = df_clean.groupby('university_tier')['cgpa'].mean()
axes[1].bar(tier_gpa.index, tier_gpa.values, color='coral', edgecolor='black')
axes[1].set_title('Средний GPA по University Tier')
axes[1].set_xlabel('University Tier')
axes[1].set_ylabel('Средний CGPA')
axes[1].set_xticks(tier_gpa.index)

plt.tight_layout()
plt.show()

corr, p = spearmanr(df_clean['university_tier'], df_clean['hired'])
print(f'Spearman (university_tier → hired): {corr:.3f}, p-value: {p:.4f}')
print('Влияет ✅' if p < 0.05 else 'Не влияет ❌')

**Результат:**
- P(hired) одинакова для всех трёх тиров — **тир университета не влияет на найм**
- Средний GPA одинаковый (~7.0) по всем тирам
- Spearman = -0.001, p = 0.67 — статистически незначимо
- Причина: датасет синтетический, `university_tier` сгенерирован независимо от найма
- **Вывод:** исключить из модели

## 8. Baseline модель (Random Forest)

Используем только IC-значимые факторы. `university_tier`, `age`, `research_papers` исключены.
Дисбаланс классов (70/30) учтён через `class_weight='balanced'`.

In [ ]:
features = [
    'experience_years', 'internships', 'skills_score',
    'projects', 'programming_languages', 'certifications',
    'hackathons', 'soft_skills_score'
]

X = df_clean[features]
y = df_clean['hired']

# stratify=y сохраняет соотношение 70/30 в train и test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train: {len(X_train)} строк')
print(f'Test:  {len(X_test)} строк')
print(f'Train hired=1: {y_train.mean():.3f}')
print(f'Test  hired=1: {y_test.mean():.3f}')

In [ ]:
# Random Forest с учётом дисбаланса классов
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'  # штраф за ошибку на меньшем классе
)
rf.fit(X_train, y_train)

y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}')

# Feature Importance
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
plt.figure(figsize=(10, 4))
importances.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Feature Importance (Random Forest)')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 9. Диагностика overfit и регуляризация

In [ ]:
# Сравниваем Train vs Test AUC — главный тест на overfit
print('=== Диагностика overfit ===')
print(f'RF базовый  — Train: {roc_auc_score(y_train, rf.predict_proba(X_train)[:,1]):.4f} | Test: {roc_auc_score(y_test, y_proba):.4f}')

# Регуляризованный RF — ограничиваем глубину
rf_reg = RandomForestClassifier(
    n_estimators=100, max_depth=5, min_samples_leaf=50,
    class_weight='balanced', random_state=42)
rf_reg.fit(X_train, y_train)
print(f'RF регуляриз— Train: {roc_auc_score(y_train, rf_reg.predict_proba(X_train)[:,1]):.4f} | Test: {roc_auc_score(y_test, rf_reg.predict_proba(X_test)[:,1]):.4f}')

# Logistic Regression как baseline
scaler = StandardScaler()
lr = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr.fit(scaler.fit_transform(X_train), y_train)
print(f'LogReg      — Train: {roc_auc_score(y_train, lr.predict_proba(scaler.transform(X_train))[:,1]):.4f} | Test: {roc_auc_score(y_test, lr.predict_proba(scaler.transform(X_test))[:,1]):.4f}')

# Cross-validation финальная оценка
scores = cross_val_score(rf_reg, X, y, cv=5, scoring='roc_auc')
print(f'\nCV ROC-AUC (5-fold): {scores.mean():.4f} ± {scores.std():.4f}')

**Диагностика:**
- **RF базовый:** Train=1.0, Test=0.518 → **overfit** — модель запомнила данные наизусть
- **RF регуляризованный:** Train≈Test≈0.557 → overfit устранён ✅
- **LogReg:** Train≈Test≈0.558 → нет overfit, но слабая модель
- **CV 0.52 ± 0.002** — стабильно около случайности на всех фолдах

**Вывод:** ROC-AUC ≈ 0.55 — потолок для этого датасета.
Причина не в модели, а в данных — `hired` сгенерирован почти случайно (max IC = 0.067).
На реальных данных ожидаемый AUC = 0.75–0.85.

## 10. Итоговые выводы

### Качество данных
- 200k строк, пропусков нет, дубликатов нет
- Исправлено: `cgpa` clip(upper=10), `resume_length_words` clip(lower=0), `university_tier` → int
- Дисбаланс классов 70/30 — учтён через `class_weight='balanced'`

### Рейтинг факторов найма (IC анализ)
| Место | Фактор | IC | Статус |
|---|---|---|---|
| 🥇 | experience_years | 0.067 | Главный предиктор |
| 🥈 | internships | 0.047 | Сильный сигнал |
| 🥉 | skills_score | 0.045 | Важен |
| 4–5 | projects, programming_languages | 0.025–0.034 | Умеренный |
| ❌ | university_tier, age, research_papers | ~0.000 | Исключить |

### Ключевые инсайты
1. **GPA бесполезен:** r(cgpa, skills) = 0.002 — работодатели смотрят на опыт
2. **Нелинейность:** 10+ проектов → P(hired) ≈ 1.0 — линейная модель не поймает
3. **Мультиколлинеарность:** projects ↔ skills_score = 0.70
4. **University tier не влияет:** синтетический датасет, все тиры одинаковые

### Результаты модели
| Модель | Train AUC | Test AUC | Вывод |
|---|---|---|---|
| RF базовый | 1.000 | 0.518 | Overfit |
| RF регуляризованный | ~0.56 | ~0.557 | ✅ Лучший |
| LogReg baseline | ~0.56 | ~0.558 | ✅ Стабилен |

### Почему AUC ≈ 0.55 — это потолок
- Max IC = 0.067 → теоретический потолок AUC ≈ 0.55–0.58
- `hired` в датасете сгенерирован случайно
- На реальных данных с IC > 0.15 ожидаемый AUC = 0.75–0.85

### Рекомендации
- Использовать **RF с max_depth=5** — устраняет overfit, лучший баланс bias/variance
- На реальных данных добавить текст резюме (NLP фичи) и результаты интервью
- Рассмотреть **LightGBM** для больших датасетов — быстрее и точнее RF